# 🌳 Module 05: Decision Trees from First Principles
## 🎓 The Complete Masterclass Guide for Data Science & Machine Learning
*Author: Muhammad Zabiullah ([@mzb000](https://github.com/mzb000))*

---

## 📌 Table of Contents
1. **What is a Decision Tree? (The Medical Diagnosis Analogy)**
2. **Purity Metrics: Shannon Entropy vs Gini Impurity**
3. **Information Gain: Deciding Where to Cut Features**
4. **Step-by-Step Hands-On Implementation from Scratch**
5. **Line-by-Line Code Breakdown**
6. **The Overfitting Hazard: Pre-Pruning vs Post-Pruning**
7. **Top Interview Questions on Trees**

---

## 💡 1. The Intuition: Asking 20 Questions
A Decision Tree breaks down complex decisions into a sequence of simple binary questions:
- "Is Glucose Level > 140?" $\to$ Yes/No
- "If Yes, is BMI > 30?" $\to$ Yes/No

The algorithm's job is to automatically find the **best feature** and **best threshold** that separates different classes with the highest purity.

## 📐 2. The Mathematics of Purity: Entropy vs Gini

### 1. Shannon Entropy (Disorder / Uncertainty):
$$H(S) = -\sum_{i=1}^{C} p_i \log_2(p_i)$$
- **Pure Node (100% Yes, 0% No):** $H(S) = -(1.0 \log_2(1.0)) = \mathbf{0.0}$ (Zero uncertainty!).
- **Maximum Chaos (50% Yes, 50% No):** $H(S) = -(0.5 \log_2(0.5) + 0.5 \log_2(0.5)) = \mathbf{1.0}$ (Maximum disorder).

### 2. Gini Impurity:
$$Gini(S) = 1 - \sum_{i=1}^{C} p_i^2$$
- Pure Node: $1 - (1.0^2 + 0.0^2) = \mathbf{0.0}$
- Maximum Chaos: $1 - (0.5^2 + 0.5^2) = \mathbf{0.5}$
- *Why does Scikit-Learn default to Gini?* Because Gini does NOT require computing logarithms ($\log_2$), making it **30% faster to train**!

In [ ]:
import numpy as np
import pandas as pd

def compute_entropy(labels):
    """Calculates Shannon Entropy for an array of class labels."""
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / len(labels)
    # Avoid log2(0) by filtering out zero probabilities
    probabilities = probabilities[probabilities > 0]
    return -np.sum(probabilities * np.log2(probabilities))

def compute_gini(labels):
    """Calculates Gini Impurity for an array of class labels."""
    _, counts = np.unique(labels, return_counts=True)
    probabilities = counts / len(labels)
    return 1.0 - np.sum(probabilities ** 2)

# Case 1: Pure Bucket (All 10 samples are Churned=1)
pure_bucket = np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

# Case 2: Mixed Bucket (5 Churned=1, 5 Retained=0)
mixed_bucket = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])

print("=== Purity Comparison ===")
print(f"Pure Bucket  -> Entropy: {compute_entropy(pure_bucket):.4f} | Gini: {compute_gini(pure_bucket):.4f}")
print(f"Mixed Bucket -> Entropy: {compute_entropy(mixed_bucket):.4f} | Gini: {compute_gini(mixed_bucket):.4f}")

## ✂️ 3. Information Gain
Information Gain measures how much disorder drops after splitting a parent node into left and right child nodes:
$$IG(S, A) = H(S) - \sum_{v \in \{L, R\}} \frac{|S_v|}{|S|} H(S_v)$$
The tree evaluates every feature and every threshold, picking the one that maximizes $IG$!

In [ ]:
def information_gain(parent, left_child, right_child):
    weight_left = len(left_child) / len(parent)
    weight_right = len(right_child) / len(parent)
    gain = compute_entropy(parent) - (weight_left * compute_entropy(left_child) + weight_right * compute_entropy(right_child))
    return gain

parent_data = np.array([1, 1, 1, 1, 0, 0, 0, 0])  # 4 Yes, 4 No (Entropy = 1.0)
split_left  = np.array([1, 1, 1, 1])               # Pure Yes (Entropy = 0.0)
split_right = np.array([0, 0, 0, 0])               # Pure No (Entropy = 0.0)

gain = information_gain(parent_data, split_left, split_right)
print(f"Parent Entropy        : {compute_entropy(parent_data):.2f}")
print(f"Information Gain (IG) : {gain:.2f} (A PERFECT SPLIT!)")

## 🎯 4. Top Interview Questions on Decision Trees
### Q1: Why do Decision Trees have high variance and overfit so easily?
- **Answer:** An unconstrained tree will continue splitting until every single leaf contains exactly 1 sample (100% training accuracy). It ends up memorizing noise instead of general patterns.
- **Remedies:**
  - max_depth: Limits tree depth.
  - min_samples_split: Minimum rows required to split.
  - min_samples_leaf: Minimum rows required in a final leaf.